### Import

In [ ]:
import os; import pandas as pd
pd.options.display.float_format = '{:.3f}'.format
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
import numpy as np; import matplotlib.pyplot as plt
import gurobipy as gp; from gurobipy import GRB
from itertools import product; from tqdm import tqdm
import importlib
import functions_utils; import functions_data
import functions_optimize; import functions_eval
importlib.reload(functions_data); importlib.reload(functions_optimize)
importlib.reload(functions_eval); importlib.reload(functions_utils)
from functions_utils import *; from functions_data import *
from functions_optimize import *; from functions_eval import *
import time

S = 15
LEVEL = "high"
SEED = 42

generation_data, I, T = load_generation_data(date_filter="2022-07-18")
R, P_RT, K, K0, M1, M2 = load_parameters(I, T, generation_data, S, LEVEL, SEED)
P_DA, P_PN = load_price_data(P_RT)

print("-"*100); print("[Individual Participation Model optimization]")
x_ind, yp_ind, ym_ind, z_ind, zc_ind, zd_ind, OBJ_IND = optimize_individually_forall(R, K, K0, P_DA, P_RT, P_PN, I, T, S, M1)

✅ 총 5개 파일을 불러왔습니다: 1201.csv, 137.csv, 401.csv, 524.csv, 89.csv
📊 데이터 Shape: I=5, T=24, S=15
✅ 시뮬레이션 초기화 완료: S=15, Randomness='high', Random Seed=1, M1=696.38, M2=1954.35
----------------------------------------------------------------------------------------------------
[Individual Participation Model optimization]


Optimizing individually for each target_i:   0%|          | 0/5 [00:00<?, ?it/s]

Set parameter Username
Set parameter LicenseID to value 2611964
Academic license - for non-commercial use only - expires 2026-01-20
Set parameter MIPGap to value 1e-07
Optimal solution found for target_i=0! Objective value: 253304.7528988332
Set parameter MIPGap to value 1e-07


Optimizing individually for each target_i:  40%|████      | 2/5 [00:00<00:00, 13.73it/s]

Optimal solution found for target_i=1! Objective value: 351023.29613336804
Set parameter MIPGap to value 1e-07
Optimal solution found for target_i=2! Objective value: 396869.8784692788
Set parameter MIPGap to value 1e-07


Optimizing individually for each target_i:  80%|████████  | 4/5 [00:00<00:00, 12.06it/s]

Optimal solution found for target_i=3! Objective value: 499705.73298714973
Set parameter MIPGap to value 1e-07


Optimizing individually for each target_i: 100%|██████████| 5/5 [00:00<00:00, 12.22it/s]

Optimal solution found for target_i=4! Objective value: 195113.49800989477


### Holistic Optimization (쌩)

In [8]:
def optimize_hol(R, K, K0, P_DA, P_RT, P_PN, I, T, S, M1):
    set = gp.Model("set_independent")
    set.setParam("MIPGap", 1e-4)

    x_hol = set.addVars(I, T, vtype=GRB.CONTINUOUS, lb=0, name="x")
    yp_hol = set.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="yp") ; ym_hol = set.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="ym")
    dp_hol = set.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="dp") ; dm_hol = set.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="dm")
    z_hol = set.addVars(I, T + 1, S, vtype=GRB.CONTINUOUS, name="z")
    zc_hol = set.addVars(I, T, S, vtype=GRB.CONTINUOUS, name="zc") ; zd_hol = set.addVars(I, T, S, vtype=GRB.CONTINUOUS, name="zd")
    phi1_hol = set.addVars(I, T, S, vtype=GRB.BINARY, name="phi1") ; phi2_hol = set.addVars(I, T, S, vtype=GRB.BINARY, name="phi2") 
    phi3_hol = set.addVars(I, T, S, vtype=GRB.BINARY, name="phi3") ; phi4_hol = set.addVars(I, T, S, vtype=GRB.BINARY, name="phi4")
    phi5_hol = set.addVars(I, T, S, vtype=GRB.BINARY, name="phi5") ; phi6_hol = set.addVars(I, T, S, vtype=GRB.BINARY, name="phi6")

    set.update()

    obj_hol = (
        gp.quicksum(P_DA[t] * x_hol[i, t] for i in range(I) for t in range(T)) +
        gp.quicksum((1/S) * (P_RT[t, s] * yp_hol[i, t, s] - P_PN[t, s] * ym_hol[i, t, s]) for i in range(I) for t in range(T) for s in range(S))
    )
    
    set.setObjective(obj_hol, GRB.MAXIMIZE)

    for i, t, s in product(range(I), range(T), range(S)):
        set.addConstr(R[i, t, s] - x_hol[i, t] == yp_hol[i, t, s] - ym_hol[i, t, s] + dp_hol[i, t, s] - dm_hol[i, t, s] + zc_hol[i, t, s] - zd_hol[i, t, s])
        set.addConstr(R[i, t, s] + zd_hol[i, t, s] >= yp_hol[i, t, s] + dp_hol[i, t, s] + zc_hol[i, t, s])
        set.addConstr(zd_hol[i, t, s] <= z_hol[i, t, s]) ; set.addConstr(zc_hol[i, t, s] <= K[i] - z_hol[i, t, s]) ; set.addConstr(z_hol[i, t, s] <= K[i])
        set.addConstr(z_hol[i, t + 1, s] == z_hol[i, t, s] + 0.92 * zc_hol[i, t, s] - zd_hol[i, t, s] / 0.95)
        
        set.addConstr(yp_hol[i, t, s] <= M1 * phi1_hol[i, t, s]) ; set.addConstr(ym_hol[i, t, s] <= M1 * (1 - phi1_hol[i, t, s]))
        set.addConstr(dp_hol[i, t, s] <= M1 * phi2_hol[i, t, s]) ; set.addConstr(dm_hol[i, t, s] <= M1 * (1 - phi2_hol[i, t, s]))
        set.addConstr(yp_hol[i, t, s] <= M1 * phi3_hol[i, t, s]) ; set.addConstr(dm_hol[i, t, s] <= M1 * (1 - phi3_hol[i, t, s]))
        set.addConstr(ym_hol[i, t, s] <= M1 * phi4_hol[i, t, s]) ; set.addConstr(dp_hol[i, t, s] <= M1 * (1 - phi4_hol[i, t, s]))
        set.addConstr(ym_hol[i, t, s] <= M1 * phi5_hol[i, t, s]) ; set.addConstr(zc_hol[i, t, s] <= M1 * (1 - phi5_hol[i, t, s]))
        set.addConstr(dm_hol[i, t, s] <= M1 * phi6_hol[i, t, s]) ; set.addConstr(zc_hol[i, t, s] <= M1 * (1 - phi6_hol[i, t, s]))

    for i, s in product(range(I), range(S)): set.addConstr(z_hol[i, 0, s] == K0[i])

    balance_constraints = {}
    for t, s in product(range(T), range(S)):
        balance_constraints[t, s] = set.addConstr(gp.quicksum(dp_hol[i, t, s] for i in range(I)) == gp.quicksum(dm_hol[i, t, s] for i in range(I)), name=f"balance_{t}_{s}")

    set.optimize()
    
    if set.status == GRB.OPTIMAL:
        print(f"Optimal solution found! Objective value: {set.objVal}")
        
        x_sol = np.array([[x_hol[i, t].X for t in range(T)] for i in range(I)])
        yp_sol = np.array([[[yp_hol[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)]) ; ym_sol = np.array([[[ym_hol[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
        dp_sol = np.array([[[dp_hol[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)]) ; dm_sol = np.array([[[dm_hol[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
        zc_sol = np.array([[[zc_hol[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)]) ; zd_sol = np.array([[[zd_hol[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
        z_sol = np.array([[[z_hol[i, t, s].X for s in range(S)] for t in range(T+1)] for i in range(I)]) ; original_objval = set.objVal
        phi1_sol = np.array([[[phi1_hol[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)]) ; phi2_sol = np.array([[[phi2_hol[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
        phi3_sol = np.array([[[phi3_hol[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)]) ; phi4_sol = np.array([[[phi4_hol[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
        phi5_sol = np.array([[[phi5_hol[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)]) ; phi6_sol = np.array([[[phi6_hol[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
        
        try:
            lambda_dual = {}
            for t, s in product(range(T), range(S)):
                lambda_dual[t, s] = balance_constraints[t, s].Pi
            print("Direct dual extraction successful!")
            
        except AttributeError:
            print("\nDirect dual extraction failed. Using Model.fixed() method...")
            
            fixed_model = set.fixed()
            fixed_model.setParam("OutputFlag", 0) ; fixed_model.setParam("MIPGap", 1e-4)
            fixed_model.optimize()
            
            if fixed_model.status == GRB.OPTIMAL:
                original_obj = set.objVal ; fixed_obj = fixed_model.objVal ; obj_diff = abs(original_obj - fixed_obj)
                print(f"Original MIP objective: {original_obj:.6f}") ; print(f"Fixed LP objective: {fixed_obj:.6f}") ; print(f"Difference: {obj_diff:.10f}")
                if obj_diff < 1e-6: print("✅ Objective values match! Fixed model is correct.")
                else: print("⚠️  Warning: Objective values don't match. Check model consistency.")
                
                print("\n=== Solution Comparison ===")
                fixed_x, fixed_yp, fixed_ym, fixed_dp, fixed_dm, fixed_zc, fixed_zd = {}, {}, {}, {}, {}, {}, {}
                
                for var in fixed_model.getVars():
                    if var.VarName.startswith('x['):
                        indices = var.VarName.replace('x[', '').replace(']', '').split(',')
                        i, t = int(indices[0]), int(indices[1])
                        fixed_x[i, t] = var.X
                    elif var.VarName.startswith('yp['):
                        indices = var.VarName.replace('yp[', '').replace(']', '').split(',')
                        i, t, s = int(indices[0]), int(indices[1]), int(indices[2])
                        fixed_yp[i, t, s] = var.X
                    elif var.VarName.startswith('ym['):
                        indices = var.VarName.replace('ym[', '').replace(']', '').split(',')
                        i, t, s = int(indices[0]), int(indices[1]), int(indices[2])
                        fixed_ym[i, t, s] = var.X
                    elif var.VarName.startswith('dp['):
                        indices = var.VarName.replace('dp[', '').replace(']', '').split(',')
                        i, t, s = int(indices[0]), int(indices[1]), int(indices[2])
                        fixed_dp[i, t, s] = var.X
                    elif var.VarName.startswith('dm['):
                        indices = var.VarName.replace('dm[', '').replace(']', '').split(',')
                        i, t, s = int(indices[0]), int(indices[1]), int(indices[2])
                        fixed_dm[i, t, s] = var.X
                    elif var.VarName.startswith('zc['):
                        indices = var.VarName.replace('zc[', '').replace(']', '').split(',')
                        i, t, s = int(indices[0]), int(indices[1]), int(indices[2])
                        fixed_zc[i, t, s] = var.X
                    elif var.VarName.startswith('zd['):
                        indices = var.VarName.replace('zd[', '').replace(']', '').split(',')
                        i, t, s = int(indices[0]), int(indices[1]), int(indices[2])
                        fixed_zd[i, t, s] = var.X
                
                max_diff_x = 0; max_diff_yp = 0; max_diff_ym = 0; max_diff_dp = 0; max_diff_dm = 0; max_diff_zc = 0; max_diff_zd = 0
                
                for i, t in product(range(I), range(T)):
                    diff = abs(x_sol[i, t] - fixed_x.get((i, t), 0)) ; max_diff_x = max(max_diff_x, diff)
                    for s in range(S):
                        diff = abs(yp_sol[i, t, s] - fixed_yp.get((i, t, s), 0)) ; max_diff_yp = max(max_diff_yp, diff)
                        diff = abs(ym_sol[i, t, s] - fixed_ym.get((i, t, s), 0)) ; max_diff_ym = max(max_diff_ym, diff)
                        diff = abs(dp_sol[i, t, s] - fixed_dp.get((i, t, s), 0)) ; max_diff_dp = max(max_diff_dp, diff)
                        diff = abs(dm_sol[i, t, s] - fixed_dm.get((i, t, s), 0)) ; max_diff_dm = max(max_diff_dm, diff)
                        diff = abs(zc_sol[i, t, s] - fixed_zc.get((i, t, s), 0)) ; max_diff_zc = max(max_diff_zc, diff)
                        diff = abs(zd_sol[i, t, s] - fixed_zd.get((i, t, s), 0)) ; max_diff_zd = max(max_diff_zd, diff)
                
                print(f"Max difference in x: {max_diff_x:.10f}"); print(f"Max difference in yp: {max_diff_yp:.10f}"); print(f"Max difference in ym: {max_diff_ym:.10f}")
                print(f"Max difference in dp: {max_diff_dp:.10f}"); print(f"Max difference in dm: {max_diff_dm:.10f}")
                print(f"Max difference in zc: {max_diff_zc:.10f}"); print(f"Max difference in zd: {max_diff_zd:.10f}")
                
                total_max_diff = max(max_diff_x, max_diff_yp, max_diff_ym, max_diff_dp, max_diff_dm, max_diff_zc, max_diff_zd)
                if total_max_diff < 1e-6: print("✅ All solutions match! Fixed model solution is correct.")
                else: print("⚠️  Warning: Solutions don't match. Check model consistency.")
            
            lambda_dual = np.zeros((T, S))

            if fixed_model.status == GRB.OPTIMAL:
                for t, s in product(range(T), range(S)):
                    constr_name = f"balance_{t}_{s}"
                    try:
                        constr = fixed_model.getConstrByName(constr_name)
                        if constr is not None: lambda_dual[t, s] = constr.Pi
                        else: lambda_dual[t, s] = np.nan
                    except: lambda_dual[t, s] = np.nan 
                        
                print("Model.fixed() dual extraction successful!")
            else:
                print("Fixed model optimization failed. Setting dual to zeros."); lambda_dual = np.zeros((T, S))

            print("\nInternal Settlement Prices (Dual Variables):")
            for t, s in product(range(T), range(0,1)):
                print(f"λ_{t}(ξ_{s}) = {lambda_dual[t, s]:.4f}")
                
    else:
        print("No optimal solution found.") ; lambda_dual = {(t, s): np.nan for t in range(T) for s in range(S)}
        x_sol = yp_sol = ym_sol = dp_sol = dm_sol = z_sol = zc_sol = zd_sol = None; original_objval = None
    
    return (x_sol, yp_sol, ym_sol, dp_sol, dm_sol, z_sol, zc_sol, zd_sol, phi1_sol, phi2_sol, phi3_sol, phi4_sol, phi5_sol, phi6_sol, original_objval, lambda_dual)

x_hol, yp_hol, ym_hol, dp_hol, dm_hol, z_hol, zc_hol, zd_hol, phi1_hol, phi2_hol, phi3_hol, phi4_hol, phi5_hol, phi6_hol, OBJ_HOL,lambda_dual = optimize_hol(R, K, K0, P_DA, P_RT, P_PN, I, T, S, M1)

Set parameter MIPGap to value 0.0001
Gurobi Optimizer version 12.0.1 build v12.0.1rc0 (mac64[arm] - Darwin 24.6.0 24G84)

CPU model: Apple M3
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 32835 rows, 23595 columns and 82875 nonzeros
Model fingerprint: 0xe9d282c1
Variable types: 12795 continuous, 10800 integer (10800 binary)
Coefficient statistics:
  Matrix range     [9e-01, 7e+02]
  Objective range  [3e+00, 2e+02]
  Bounds range     [1e+00, 1e+00]
  RHS range        [8e-02, 7e+02]
Presolve removed 11080 rows and 6648 columns
Presolve time: 0.09s
Presolved: 21755 rows, 16947 columns, 57529 nonzeros
Variable types: 9267 continuous, 7680 integer (7680 binary)
Found heuristic solution: objective 1781618.3449

Root relaxation: objective 1.809765e+06, 12008 iterations, 0.26 seconds (0.35 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/N

In [3]:
case_found = [False] * 7
for i, t, s in product(range(I), range(T), range(S)):
    if not case_found[0] and yp_hol[i, t, s] > 0 and ym_hol[i, t, s] > 0:
        print(f"Case 1 발생: i={i}, t={t}, s={s} - yp_hol={yp_hol[i,t,s]}, ym_hol={ym_hol[i,t,s]}") ; case_found[0] = True
    if not case_found[1] and ym_hol[i, t, s] > 0 and zc_hol[i, t, s] > 0:
        print(f"Case 2 발생: i={i}, t={t}, s={s} - ym_hol={ym_hol[i,t,s]}, zc_hol={zc_hol[i,t,s]}") ; case_found[1] = True
    if not case_found[2] and zc_hol[i, t, s] > 0 and zd_hol[i, t, s] > 0:
        print(f"Case 3 발생: i={i}, t={t}, s={s} - zc_hol={zc_hol[i,t,s]}, zd_hol={zd_hol[i,t,s]}") ; case_found[2] = True
    if not case_found[3] and dp_hol[i, t, s] > 0 and dm_hol[i, t, s] > 0:
        print(f"Case 4 발생: i={i}, t={t}, s={s} - dp_hol={dp_hol[i,t,s]}, dm_hol={dm_hol[i,t,s]}") ; case_found[3] = True
    if not case_found[4] and zc_hol[i, t, s] > 0 and dm_hol[i, t, s] > 0:
        print(f"Case 5 발생: i={i}, t={t}, s={s} - zc_hol={zc_hol[i,t,s]}, dm_hol={dm_hol[i,t,s]}") ; case_found[4] = True
    if not case_found[5] and yp_hol[i, t, s] > 0 and dm_hol[i, t, s] > 0:
        print(f"Case 6 발생: i={i}, t={t}, s={s} - yp_hol={yp_hol[i,t,s]}, dm_hol={dm_hol[i,t,s]}") ; case_found[5] = True
    if not case_found[6] and ym_hol[i, t, s] > 0 and dp_hol[i, t, s] > 0:
        print(f"Case 7 발생: i={i}, t={t}, s={s} - ym_hol={ym_hol[i,t,s]}, dp_hol={dp_hol[i,t,s]}") ; case_found[6] = True
    if all(case_found): break
for i, found in enumerate(case_found, 1):
    if not found: print(f"Case {i}: 발생하지 않음")

Case 1: 발생하지 않음
Case 2: 발생하지 않음
Case 3: 발생하지 않음
Case 4: 발생하지 않음
Case 5: 발생하지 않음
Case 6: 발생하지 않음
Case 7: 발생하지 않음


In [4]:
header = (
    f"{'t':>2} | "
    f"{'R':>8} {'x':>8} {'y+':>8} {'y-':>8} "
    f"{'d+':>8} {'d-':>8} {'zc':>8} {'zd':>8} {'z':>8}\n"
    + "-" * 90
)
print(header)

for t in range(8, 22):
    # 각 변수의 시나리오 평균 계산
    R_avg = np.mean([R[:, t, s].sum() for s in range(S)])
    x_sum = x_hol[:, t].sum()
    yp_avg = np.mean([yp_hol[:, t, s].sum() for s in range(S)])
    ym_avg = np.mean([ym_hol[:, t, s].sum() for s in range(S)])
    dp_avg = np.mean([dp_hol[:, t, s].sum() for s in range(S)])
    dm_avg = np.mean([dm_hol[:, t, s].sum() for s in range(S)])
    zc_avg = np.mean([zc_hol[:, t, s].sum() for s in range(S)])
    zd_avg = np.mean([zd_hol[:, t, s].sum() for s in range(S)])
    z_avg = np.mean([z_hol[:, t, s].sum() for s in range(S)])
    
    print(
        f"{t:>2} | "
        f"{R_avg:>8.2f} {x_sum:>8.2f} {yp_avg:>8.2f} {ym_avg:>8.2f} "
        f"{dp_avg:>8.2f} {dm_avg:>8.2f} {zc_avg:>8.2f} {zd_avg:>8.2f} {z_avg:>8.2f}"
    )

 t |        R        x       y+       y-       d+       d-       zc       zd        z
------------------------------------------------------------------------------------------
 8 |    40.96     0.00     6.60     0.00     0.00     0.00    35.14     0.77    15.58
 9 |   190.42    56.12     0.38     0.00    53.43    53.43   133.93     0.00    47.09
10 |   420.17   233.72    76.89     0.00    57.85    57.85   112.93     3.37   170.31
11 |   598.13   317.36   153.59     0.00    55.52    55.52   152.30    25.13   270.66
12 |   853.35     0.00   742.64     0.00     0.00     0.00   110.71     0.00   384.32
13 |  1300.22     0.00  1596.78     0.00     0.00     0.00     2.12   298.68   486.17
14 |  1405.20   917.82   216.57     0.00   574.30   574.30   276.09     5.28   173.72
15 |   825.71     0.00  1226.21     0.00     0.00     0.00     0.00   400.50   422.17
16 |   759.03   523.18   107.36     8.85   226.86   226.86   137.34     0.00     0.59
17 |   820.32     0.00   933.22     0.00     0.00

### Individual Replay

In [ ]:
def individual_replay(R, K, K0, P_DA, P_RT, P_PN, I, T, S, M1, lambda_dual, phi1_hol, phi2_hol, phi3_hol, phi4_hol, phi5_hol, phi6_hol):
    
    model = gp.Model("DER_Individual_Replay")
    model.setParam("MIPGap", 1e-4)
    
    x = model.addVars(I, T, vtype=GRB.CONTINUOUS, lb=0, name="x")
    yp = model.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="yp") ; ym = model.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="ym") 
    dp = model.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="dp") ; dm = model.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="dm") 
    z = model.addVars(I, T + 1, S, vtype=GRB.CONTINUOUS, lb=0, name="z")
    zc = model.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="zc") ; zd = model.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="zd")
    
    model.update()

    obj = (
        gp.quicksum(P_DA[t] * x[i, t] for i in range(I) for t in range(T)) + 
        gp.quicksum((1/S) * (
            P_RT[t, s] * yp[i, t, s] - P_PN[t, s] * ym[i, t, s]
        ) for i in range(I) for t in range(T) for s in range(S)) +
        gp.quicksum(lambda_dual[t, s] * (
            gp.quicksum(dm[i, t, s] for i in range(I)) - gp.quicksum(dp[i, t, s] for i in range(I))
        ) for t in range(T) for s in range(S))
    )
    
    model.setObjective(obj, GRB.MAXIMIZE)
    
    for i, t, s in product(range(I), range(T), range(S)):
        model.addConstr(R[i, t, s] - x[i, t] == yp[i, t, s] - ym[i, t, s] + dp[i, t, s] - dm[i, t, s] + zc[i, t, s] - zd[i, t, s])
        model.addConstr(R[i, t, s] + zd[i, t, s] >= yp[i, t, s] + dp[i, t, s] + zc[i, t, s])
        model.addConstr(zd[i, t, s] <= z[i, t, s]) ; model.addConstr(zc[i, t, s] <= K[i] - z[i, t, s]) ; model.addConstr(z[i, t, s] <= K[i])
        model.addConstr(z[i, t + 1, s] == z[i, t, s] + 0.92 * zc[i, t, s] - zd[i, t, s] / 0.95)
        
        model.addConstr(yp[i, t, s] <= M1 * phi1_hol[i, t, s]) ; model.addConstr(ym[i, t, s] <= M1 * (1 - phi1_hol[i, t, s]))
        model.addConstr(dp[i, t, s] <= M1 * phi2_hol[i, t, s]) ; model.addConstr(dm[i, t, s] <= M1 * (1 - phi2_hol[i, t, s]))
        model.addConstr(yp[i, t, s] <= M1 * phi3_hol[i, t, s]) ; model.addConstr(dm[i, t, s] <= M1 * (1 - phi3_hol[i, t, s]))
        model.addConstr(ym[i, t, s] <= M1 * phi4_hol[i, t, s]) ; model.addConstr(dp[i, t, s] <= M1 * (1 - phi4_hol[i, t, s]))
        model.addConstr(ym[i, t, s] <= M1 * phi5_hol[i, t, s]) ; model.addConstr(zc[i, t, s] <= M1 * (1 - phi5_hol[i, t, s]))
        model.addConstr(dm[i, t, s] <= M1 * phi6_hol[i, t, s]) ; model.addConstr(zc[i, t, s] <= M1 * (1 - phi6_hol[i, t, s]))

    for i, s in product(range(I), range(S)): model.addConstr(z[i, 0, s] == K0[i])

    model.optimize()
    
    if model.status == GRB.OPTIMAL: print(f"Optimal solution found! Objective value: {model.objVal}")
    else: print("No optimal solution found.")
    
    x_sol = np.array([[x[i, t].X for t in range(T)] for i in range(I)])
    yp_sol = np.array([[[yp[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
    ym_sol = np.array([[[ym[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
    dp_sol = np.array([[[dp[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
    dm_sol = np.array([[[dm[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
    z_sol = np.array([[[z[i, t, s].X for s in range(S)] for t in range(T+1)] for i in range(I)])
    zc_sol = np.array([[[zc[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
    zd_sol = np.array([[[zd[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
    
    return (x_sol, yp_sol, ym_sol, z_sol, zc_sol, zd_sol, dp_sol, dm_sol, model.objVal)

In [6]:
x_re, yp_re, ym_re, z_re, zc_re, zd_re, dp_re, dm_re, OBJ_RE = individual_replay(R, K, K0, P_DA, P_RT, P_PN, I, T, S, M1, lambda_dual, phi1_hol, phi2_hol, phi3_hol, phi4_hol, phi5_hol, phi6_hol)

Set parameter TimeLimit to value 900
Gurobi Optimizer version 12.0.1 build v12.0.1rc0 (mac64[arm] - Darwin 24.6.0 24G84)

CPU model: Apple M3
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Non-default parameters:
TimeLimit  900

Optimize a model with 32475 rows, 12795 columns and 57675 nonzeros
Model fingerprint: 0xf0bf5190
Coefficient statistics:
  Matrix range     [9e-01, 1e+00]
  Objective range  [3e+00, 2e+02]
  Bounds range     [0e+00, 0e+00]
  RHS range        [8e-02, 7e+02]
Presolve removed 27840 rows and 8656 columns
Presolve time: 0.01s
Presolved: 4635 rows, 4139 columns, 18514 nonzeros

Concurrent LP optimizer: primal simplex, dual simplex, and barrier
Showing barrier log only...

Ordering time: 0.01s

Barrier statistics:
 AA' NZ     : 4.012e+04
 Factor NZ  : 1.187e+05 (roughly 5 MB of memory)
 Factor Ops : 5.434e+06 (less than 1 second per iteration)
 Threads    : 6

                  Objective                Residual
Iter       Primal          

In [7]:
header = (
    f"{'t':>2} | "
    f"{'R':>8} {'x':>8} {'y+':>8} {'y-':>8} "
    f"{'d+':>8} {'d-':>8} {'zc':>8} {'zd':>8} {'z':>8}\n"
    + "-" * 90
)
print("\n==============REPLAY==============") ; print(header)

for t in range(8, 22):
    # 각 변수의 시나리오 평균 계산
    R_avg = np.mean([R[:, t, s].sum() for s in range(S)]) ; x_sum = x_re[:, t].sum()
    yp_avg = np.mean([yp_re[:, t, s].sum() for s in range(S)]) ; ym_avg = np.mean([ym_re[:, t, s].sum() for s in range(S)])
    dp_avg = np.mean([dp_re[:, t, s].sum() for s in range(S)]) ; dm_avg = np.mean([dm_re[:, t, s].sum() for s in range(S)])
    zc_avg = np.mean([zc_re[:, t, s].sum() for s in range(S)]) ; zd_avg = np.mean([zd_re[:, t, s].sum() for s in range(S)]) ; z_avg = np.mean([z_re[:, t, s].sum() for s in range(S)])
    
    print(
        f"{t:>2} | "
        f"{R_avg:>8.2f} {x_sum:>8.2f} {yp_avg:>8.2f} {ym_avg:>8.2f} "
        f"{dp_avg:>8.2f} {dm_avg:>8.2f} {zc_avg:>8.2f} {zd_avg:>8.2f} {z_avg:>8.2f}"
    )

header = (
    f"{'t':>2} | "
    f"{'R':>8} {'x':>8} {'y+':>8} {'y-':>8} "
    f"{'d+':>8} {'d-':>8} {'zc':>8} {'zd':>8} {'z':>8}\n"
    + "-" * 90
)
print("\n==============HOLISTIC==============") ; print(header)

for t in range(8, 22):
    # 각 변수의 시나리오 평균 계산
    R_avg = np.mean([R[:, t, s].sum() for s in range(S)]) ; x_sum = x_hol[:, t].sum()
    yp_avg = np.mean([yp_hol[:, t, s].sum() for s in range(S)]) ; ym_avg = np.mean([ym_hol[:, t, s].sum() for s in range(S)])
    dp_avg = np.mean([dp_hol[:, t, s].sum() for s in range(S)]) ; dm_avg = np.mean([dm_hol[:, t, s].sum() for s in range(S)])
    zc_avg = np.mean([zc_hol[:, t, s].sum() for s in range(S)]) ; zd_avg = np.mean([zd_hol[:, t, s].sum() for s in range(S)]) ; z_avg = np.mean([z_hol[:, t, s].sum() for s in range(S)])
    
    print(
        f"{t:>2} | "
        f"{R_avg:>8.2f} {x_sum:>8.2f} {yp_avg:>8.2f} {ym_avg:>8.2f} "
        f"{dp_avg:>8.2f} {dm_avg:>8.2f} {zc_avg:>8.2f} {zd_avg:>8.2f} {z_avg:>8.2f}"
    )


==============REPLAY==============
 t |        R        x       y+       y-       d+       d-       zc       zd        z
------------------------------------------------------------------------------------------
 8 |    40.96     0.00     6.88     0.00     0.00     0.00    34.79     0.71    14.70
 9 |   190.42     5.34     3.74     0.00    82.06     2.66   101.93     0.00    45.96
10 |   420.17   262.39    71.18     0.00    66.42    77.68   100.00     2.14   139.73
11 |   598.13   317.36   126.21     0.00    70.35    57.32   149.65     8.12   229.47
12 |   853.35     0.00   735.92     0.00     0.00     0.00   117.43     0.00   358.60
13 |  1300.22     0.00  1566.59     0.00     0.00     0.00     2.00   268.37   466.64
14 |  1405.20   868.31   309.24     0.00   539.66   575.83   263.83     0.00   185.98
15 |   825.71     0.00  1227.14     0.00     0.00     0.00     0.00   401.43   428.70
16 |   759.03   356.54    65.75     0.40   285.47    54.40   106.08     0.00     6.14
17 |   820.32